In [1]:
COLUMNS = ["icao","takeoff_time","landing_time","takeoff_time_df2","landing_time_df2", "match_status"]

# Do join where my model flights are right for an icao but adsbx is wrong.

In [2]:
# I can use visualize.

In [3]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights.flights_match_adsb import get_matching_icaos_in_flights
from flights_ai_model.validation import flights_df_comparison
test_date = date(2026,3,1)
df_gold = get_sfdps_flights_day(test_date)
df_adsb = read_adsb(test_date)
df_gold = get_matching_icaos_in_flights(df_gold, df_adsb)

In [4]:
df_test = get_flights(test_date)
df_model = df_test
expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
df_test = df_test.filter(expr)
df_gold = df_gold.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao").implode()))
df_model_flights, stats = flights_df_comparison(df_test,df_gold)
stats


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 3873
Predicted flights: 7912  |  Gold flights: 7768
Correctly matched: 7427  |  Predicted unmatched: 485  |  Gold flights missed: 341
Precision: 93.87%  |  Recall: 95.61%  |  F1: 94.73%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 6852/7427 matched (92.26%)
landing_airport_ident: 6682/7427 matched (89.97%)
overall: 13534/14854 matched (91.11%)


{'n_common_icaos': 3873,
 'n_predicted': 7912,
 'n_gold': 7768,
 'n_correctly_matched': 7427,
 'n_predicted_unmatched': 485,
 'n_gold_missed': 341,
 'precision': 0.9387007077856421,
 'recall': 0.956101956745623,
 'f1': 0.9473214285714285,
 'airport_ident_pct_match': 0.9111350477985728}

In [5]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights.flights_match_adsb import get_matching_icaos_in_flights
from flights_ai_model.validation import flights_df_comparison
df_test = get_adsbx_flights_for_day(test_date)
df_adsbx = df_test
expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
df_test = df_test.filter(expr)
df_gold = df_gold.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao").implode()))
df_adsbx_flights, stats = flights_df_comparison(df_test,df_gold)
stats


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 3481
Predicted flights: 7155  |  Gold flights: 7768
Correctly matched: 6954  |  Predicted unmatched: 201  |  Gold flights missed: 814
Precision: 97.19%  |  Recall: 89.52%  |  F1: 93.20%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 6843/6954 matched (98.40%)
landing_airport_ident: 6691/6954 matched (96.22%)
overall: 13534/13908 matched (97.31%)


{'n_common_icaos': 3481,
 'n_predicted': 7155,
 'n_gold': 7768,
 'n_correctly_matched': 6954,
 'n_predicted_unmatched': 201,
 'n_gold_missed': 814,
 'precision': 0.9719077568134172,
 'recall': 0.895211122554068,
 'f1': 0.9319841854854923,
 'airport_ident_pct_match': 0.9731090020132298}

In [6]:
df_model_flights = df_model_flights.select("icao", "takeoff_time", "landing_time", "takeoff_time_df2", "landing_time_df2", "match_status")
df_adsbx_flights = df_adsbx_flights.select("icao", "takeoff_time", "landing_time", "takeoff_time_df2", "landing_time_df2", "match_status")

# Do join where adsbx is right but model flights is wrong

In [7]:
def icaos_where_left_all_matched_right_not_all_matched(
    df_left,
    df_right,
    *,
    icao_col="icao",
    status_col="match_status",
    matched_value="matched",
):
    left_all_matched_icaos = (
        df_left
        .group_by(icao_col)
        .agg((pl.col(status_col) == matched_value).all().alias("all_matched"))
        .filter(pl.col("all_matched"))
        .select(icao_col)
    )

    right_not_all_matched_icaos = (
        df_right
        .group_by(icao_col)
        .agg((pl.col(status_col) == matched_value).all().alias("all_matched"))
        .filter(~pl.col("all_matched"))
        .select(icao_col)
    )

    return (
        left_all_matched_icaos
        .join(right_not_all_matched_icaos, on=icao_col, how="inner")
        .get_column(icao_col)
    )

icaos_model_matched_adsbx_not = icaos_where_left_all_matched_right_not_all_matched(
    df_model_flights,
    df_adsbx_flights,
)
icaos_adsbx_matched_model_not = icaos_where_left_all_matched_right_not_all_matched(
    df_adsbx_flights,
    df_model_flights,
)

In [8]:
icao = icaos_adsbx_matched_model_not.to_list()[0]
display(df_adsbx.filter(pl.col("icao") == icao))
display(df_model.filter(pl.col("icao") == icao))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool
"""a7e18a""",null,"""N607CZ""",2026-02-28 23:48:02,"""KOMA""",2026-03-01 02:08:34,"""KSLC""",false,false,false,false
"""a7e18a""",null,"""N607CZ""",2026-03-01 21:09:23,"""KSLC""",2026-03-01 23:16:24,"""KXNA""",false,false,false,false


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a7e18a""","""SKW3620""","""N607CZ""",2026-03-01 22:45:30.720,"""US-5178""",2026-03-01 23:15:35.290,"""KXNA""",false,false,false,false,"""E75L""","""DELTA AIR LINES INC""","""EMBRAER ERJ-170-200 (long wing…","""A3"""


In [10]:
from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from evaluation.flights.visual import visualize_flight

visualize_flight(df_model.filter(pl.col("icao") == icao), read_icaos_from_adsb(test_date, [icao]))
5+5

10